In [1]:
# Import packages here
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy import integrate
import scipy.stats as ss
from scipy.interpolate import UnivariateSpline
from scipy.stats import poisson
from multiprocessing import Pool
import time
import multiprocessing as mp
import pandas as pd
from scipy.interpolate import interp1d
from astropy.io import fits
from tqdm import tqdm
import functools
%matplotlib notebook

In [2]:
###########################################################################
# Some constants
c = 2.99792458*10**(10)			# cm/s		 
h = 6.6260755*10**(-27)			# erg s
k = 1.380658*10**(-16)				# erg/K
n_avogadro = 6.02214179*10**(23)      	        # Avogadro's number
L_sun = 3.8458*10**(33)			# Units: erg/s
stefan_sigma = 5.67051*10**(-5) 	        # Units: erg.cm^(-2).K^(-4).s^(-1)
pi = math.pi
###########################################################################

In [3]:
###########################################################################
# Function to calculate the specific heat of a molecule
def specific_heat(Temperature_array):
	specific_heat_T = np.array([])
	def planck_sp(T):
		frac = h*freq_fundamental_modes/(k*T)
		frac = frac.astype(np.longdouble)
		expr_a = 1/np.exp(frac/2)
		expr_b = (expr_a)**2
		expr_1 = np.exp(frac)/(frac**2)
		expr_2 = expr_b/(frac**2)
		expr_3 = 2/(frac**2)
		expr_4 = expr_1 + expr_2 - expr_3
		expr_5 = 1/expr_4
		expr = expr_5
		#expr = (frac**2)*np.exp(frac)/((np.exp(frac)-1)**2)
		specific_heat_val = (np.sum(expr))*k
		return specific_heat_val
	specific_heat_T = list(map(planck_sp, Temperature_array))
	specific_heat_T = np.array(specific_heat_T)
	return specific_heat_T
###########################################################################


###########################################################################
def cooling_curve(Temperature_array):
	a = specific_heat(Temperature_array)
	term = 4*pi/(a)
	def planck(nu, sig):
		frac = h*nu/(k*Temperature_array)
		frac = frac.astype(np.longdouble)
		expr_a = 1/(np.exp(frac)-1)
		expr_b = 2*h*(nu**3)/(c**2)
		expr = expr_a*expr_b*sig 
		return expr
	sigma_B_nu_T = list(map(planck, freq_IR_active_modes, xsection))
	sigma_B_nu_T = np.array(sigma_B_nu_T)
	for i in range (0, len(xsection)):
		a = sigma_B_nu_T[i]
		if i == 0:
			b = a
			cooling_curve_T = b
		else:
			b = cooling_curve_T+a
			cooling_curve_T = b
			
	cooling_curve_T = term*cooling_curve_T
	return cooling_curve_T
###########################################################################

###########################################################################
# Function to calculate rbar
def rbar(Temp_star, G_0):
	freq_photon_energy = energy_absorption_x_section*(1.60218*10**(-12))/h 
	Temp_star = float(Temp_star)
	frac = h*freq_photon_energy/(k*Temp_star)
	frac = frac.astype(np.longdouble)
	expr_a = 1/(np.exp(frac)-1)
	expr_b = 2*h*(freq_photon_energy**3)/(c**2)
	B_nu_array = expr_a*expr_b
	mean_photon_intensity = B_nu_array/(h*freq_photon_energy)
	
	energy_array_FUV = np.linspace(6, 13.6, 1000)
	frequency_array_FUV = energy_array_FUV*(1.60218*10**(-12))/h
	frac = h*frequency_array_FUV/(k*Temp_star)
	frac = frac.astype(np.longdouble)
	expr_a = 1/(np.exp(frac)-1)
	expr_b = 2*h*(frequency_array_FUV**3)/(c**2)
	B_nu_FUV = expr_a*expr_b
	Total_flux = stefan_sigma*(Temp_star)**(4)
	Flux_FUV = (pi)*(integrate.simps(B_nu_FUV, frequency_array_FUV))
	frac_FUV = Flux_FUV/Total_flux
	Field_dilution_factor = 1.6*10**(-3)*(G_0)/(stefan_sigma*((Temp_star)**(4))*frac_FUV)
	
	expr = pi*Field_dilution_factor*absorption_x_section*mean_photon_intensity
	rbar_calc = (integrate.simps(expr, freq_photon_energy))
	return rbar_calc

###########################################################################
# for each temp value, lets create a temperature_grid 
def create_temp_grid(T_max, T_min):
    diff = T_max - T_min
    if diff < 50:
        temp_array = np.arange(T_min, T_max, 0.01)
    else:
        temp_array = np.arange(T_min, T_max, 1)    
        
    return temp_array

def expr_tau_min_calc(array):
        return 1/array
    
def calculate_tau_min(expr, t_array):
        tau_min_val = integrate.simps(expr,t_array)
        return tau_min_val

In [4]:

###########################################################################
# Function to calculate the maximum temperature that a molecule reaches upon absorption of a UV photon
def Temp_max(energy, fine_tune = False):
    # Initialize temperature list with the background temperature of 2.7 K
    temp_list = [2.7]
    energy_calc_eV = 0.0
    step = 10  # Start with a larger step for faster convergence
    energy_list = [0.0]

    # Loop to increase temperature until reaching the energy target
    while energy_calc_eV < energy:
        specific_heat_T_new = specific_heat(np.array(temp_list))
        energy_calc = integrate.simps(specific_heat_T_new, temp_list)
        energy_calc_eV = energy_calc / (1.6021772 * 10**(-12))

        if energy_calc_eV < energy:
            temp_list.append(temp_list[-1] + step)
            energy_list.append(energy_calc_eV)

            
    # Fine-tune the temperature with smaller steps if needed
    if fine_tune:
        step = 0.1
        while abs(energy_calc_eV - energy) > 0.01:  # Adjust tolerance as necessary
            specific_heat_T_new = specific_heat(np.array(temp_list))
            energy_calc = integrate.simps(specific_heat_T_new, temp_list)
            energy_calc_eV = energy_calc / (1.6021772 * 10**(-12))

            # Adjust temperature based on whether we are above or below the energy
            if energy_calc_eV > energy:
                temp_list.append(temp_list[-1] - step)
            else:
                temp_list.append(temp_list[-1] + step)

    # Final temperature and array
    T_max = temp_list[-1]
    
    return T_max, temp_list, energy_list


def calculate_G_T(row):
    T = row['new_T_grid']
    T_max_val = row['T_max']
    print (T_max_val)
  
     
    
    temp_grid_partial = functools.partial(create_temp_grid, T_max_val)
    temp_grids_for_tau_min = np.array(list(map(temp_grid_partial, T)))
    
    # lets calculate the time it takes for a molecule to cool from T_max_val
    cc_t = list(map(cc_func, temp_grids_for_tau_min)) #cooling curves
    expr_t = list(map(expr_tau_min_calc, cc_t)) # 1/cooling curve
    tau_min_t = list(map(calculate_tau_min, expr_t, temp_grids_for_tau_min))
    tau_min_t = np.array(tau_min_t)

    
    rate_param = rbar_calc*tau_min_t
    exp_fac = np.exp(-rate_param)
    G_T_num = rbar_calc*exp_fac
    G_T_denom = cc_func(T)
    G_1_T = G_T_num/G_T_denom
    int_number = integrate.simps(G_1_T, T)

    return G_1_T, int_number


# Function to calculate the intensities of IR active modes
def intensities_nu_erg_per_s(G_T, Temp_array):
    intensity_nu = np.array([])
    for i in range(0, len(xsection)):
        G_T_B_nu_T = np.array([])
        frac = h*freq_IR_active_modes[i]/(k*Temp_array)
        frac = frac.astype(np.longdouble)
        expr_a = 1/(np.exp(frac)-1)
        expr_b = 2*h*(freq_IR_active_modes[i]**3)/(c**2)
        expr = expr_a*expr_b
        G_T_B_nu_val = G_T*expr
        G_T_B_nu_T = np.append(G_T_B_nu_T, G_T_B_nu_val)
        integral = integrate.simps(G_T_B_nu_T, Temp_array)
        intensity_nu_val = xsection[i]*integral
        intensity_nu = np.append(intensity_nu, intensity_nu_val)
    return intensity_nu

In [5]:
###########################################################################
if __name__ == "__main__":
    start_time = time.time()
    Temp_star = float(15000) #Temperature of central star of NGC 7023; Units: K
    G_0 = 10000
    
    # Some infor for the molecule
    energy_absorption_x_section = np.loadtxt("abs_cross-section_coronene_neutral.csv",
                                             delimiter = ',', skiprows = 1, usecols = (0))
    ind = np.where(energy_absorption_x_section < 13.6)
    ind = ind[0]
    absorption_x_section = np.loadtxt("abs_cross-section_coronene_neutral.csv",
                                      delimiter = ',', skiprows = 1, usecols = (1))
    energy_absorption_x_section = energy_absorption_x_section[ind]
    absorption_x_section = absorption_x_section[ind] #Units: Mb
    absorption_x_section = absorption_x_section*10**(-18) #Units: cm^(2)
    
    ###########################################################################
    num_c_atoms = 18
    correction_factor = 1#0.66
    print ('Correction factor is', correction_factor)
    ###########################################################################
    # Import Data here
    data_C60 = np.loadtxt("coronene_neutral_IR_active.csv", 
                          delimiter = ',', skiprows = 1, usecols = (0,1)) 

    wno_IR_active_modes = data_C60[:,0] # Units: cm^(-1)
    ind = np.where((wno_IR_active_modes>2750)&(wno_IR_active_modes<3250))
    ind = ind[0]
    freq_IR_active_modes = c*wno_IR_active_modes # units: Hz 
    
    wno_fundamental_modes = np.loadtxt("coronene_neutral.csv",
                                       delimiter = ',', skiprows = 1,  usecols = (0)) # Units: cm^(-1)

    freq_fundamental_modes = c*wno_fundamental_modes # Units: Hz 

    int_intensity = data_C60[:,1] 	# Units: km/mol
    xsection = int_intensity*c*(10**5)/n_avogadro 	# Units: cm^(2).Hz
    xsection[ind] = xsection[ind]*correction_factor
    ###########################################################################
    
    # First we will calculate the maximum temperature a molecule will reach if it absorbs 13.6 eV photon
    T = Temp_max(13.6, fine_tune = False)
    
    # Lets define a temperature grid
    T_min = 1 #K
    T_max = T[0] - 0.1
    T_grid = np.arange(T_min, T_max, 0.1)
    
    # Calculate specific heat
    cv_grid = specific_heat(T_grid)
    
    # calculate cooling curve
    cc_grid = cooling_curve(T_grid)
    
    # for use in the calculations
    # convert cc_grid into a function
    cc_func = interp1d(T_grid, cc_grid, fill_value = 'extrapolate')
    
     # calculate rbar
    rbar_calc = rbar(Temp_star, G_0)
    
    # calculate G_T
    G_T = pd.DataFrame({'E_p': T[2],
                      'T_max': T[1]})
    G_T_subset = G_T[G_T['E_p'] != 0]
    
    G_T_subset['T_grid'] = [T_grid]*len(G_T_subset)
    G_T_subset['new_T_grid'] = G_T_subset.apply(lambda row: [t for t in row['T_grid'] if t< row['T_max']],
                                                axis = 1)
    G_T_subset = G_T_subset[['E_p', 'T_max', 'new_T_grid']]
    print (G_T_subset)
    
    G_T_subset = G_T_subset.iloc[[-1]]
    results = G_T_subset.apply(calculate_G_T, axis = 1)
    G_T_subset['G_1_T'], G_T_subset['int_results'] = zip(*results)
    print (G_T_subset)
    
    G_T = np.array(G_T_subset['G_1_T'].tolist()[0])
    Temp_array = np.array(G_T_subset['new_T_grid'].tolist()[0])
    intensity_array = intensities_nu_erg_per_s(G_T, Temp_array)
    print (intensity_array)
        
    print ("My program took", time.time() - start_time, "to run")
   
      
    


Correction factor is 1
(2242.7, [2.7, 12.7, 22.7, 32.7, 42.7, 52.7, 62.7, 72.7, 82.7, 92.7, 102.7, 112.7, 122.7, 132.7, 142.7, 152.7, 162.7, 172.7, 182.7, 192.7, 202.7, 212.7, 222.7, 232.7, 242.7, 252.7, 262.7, 272.7, 282.7, 292.7, 302.7, 312.7, 322.7, 332.7, 342.7, 352.7, 362.7, 372.7, 382.7, 392.7, 402.7, 412.7, 422.7, 432.7, 442.7, 452.7, 462.7, 472.7, 482.7, 492.7, 502.7, 512.7, 522.7, 532.7, 542.7, 552.7, 562.7, 572.7, 582.7, 592.7, 602.7, 612.7, 622.7, 632.7, 642.7, 652.7, 662.7, 672.7, 682.7, 692.7, 702.7, 712.7, 722.7, 732.7, 742.7, 752.7, 762.7, 772.7, 782.7, 792.7, 802.7, 812.7, 822.7, 832.7, 842.7, 852.7, 862.7, 872.7, 882.7, 892.7, 902.7, 912.7, 922.7, 932.7, 942.7, 952.7, 962.7, 972.7, 982.7, 992.7, 1002.7, 1012.7, 1022.7, 1032.7, 1042.7, 1052.7, 1062.7, 1072.7, 1082.7, 1092.7, 1102.7, 1112.7, 1122.7, 1132.7, 1142.7, 1152.7, 1162.7, 1172.7, 1182.7, 1192.7, 1202.7, 1212.7, 1222.7, 1232.7, 1242.7, 1252.7, 1262.7, 1272.7, 1282.7, 1292.7, 1302.7, 1312.7, 1322.7, 1332.7, 1342.7

<ipython-input-11-b53245c20e7a>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  G_T_subset['T_grid'] = [T_grid]*len(G_T_subset)


           E_p   T_max                                         new_T_grid
2     0.000004    22.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
3     0.000089    32.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
4     0.000563    42.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
5     0.001554    52.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
6     0.003104    62.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
..         ...     ...                                                ...
220  13.224647  2202.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
221  13.306970  2212.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
222  13.389340  2222.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
223  13.471753  2232.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...
224  13.554211  2242.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...

[223 rows x 3 columns]


<ipython-input-11-b53245c20e7a>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  G_T_subset['new_T_grid'] = G_T_subset.apply(lambda row: [t for t in row['T_grid'] if t< row['T_max']],


2242.7


<ipython-input-4-06c6a75dec2f>:49: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  temp_grids_for_tau_min = np.array(list(map(temp_grid_partial, T)))


           E_p   T_max                                         new_T_grid  \
224  13.554211  2242.7  [1.0, 1.1, 1.2000000000000002, 1.3000000000000...   

                                                 G_1_T  int_results  
224  (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...     0.999904  


[1.38476742e-18 1.86508602e-18 1.86520459e-18 2.16141332e-17
 5.88895713e-18 5.88835237e-18 1.51364214e-19 1.51364214e-19
 1.92830024e-16 8.69786812e-18 8.69679975e-18 1.39299672e-18
 1.39271856e-18 3.50294977e-17 3.50340172e-17 1.27837056e-18
 1.27852100e-18 2.12846691e-18 2.12893629e-18 2.11985081e-17
 2.11947887e-17 1.39574329e-17 1.39583088e-17 2.45143453e-16
 2.45147299e-16]
